In [ ]:
import serial
import csv
import time
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import find_peaks

# Step 1: Configure Serial Communication
esp32_port = "/dev/tty.usbserial-1420"  # Change this to your ESP32's port
baud_rate = 115200
SAMPLES = 1024  # Must match ESP32 SAMPLES
fs = 125  # Sampling frequency in Hz
timeout = 2  # Timeout for serial communication in seconds

try:
    ser = serial.Serial(esp32_port, baud_rate, timeout=timeout)
    print(f"Serial communication established on {esp32_port} with baud rate {baud_rate}")
except Exception as e:
    print(f"Failed to open serial port {esp32_port}: {e}")
    exit()

# Step 2: Read ECG Data from CSV
input_csv = r"/Users/sarahalabdulrazzak/Desktop/Capstone/myenv/Data/mimic_perform_non_af_csv/mimic_perform_non_af_009_data.csv"
print(f"Reading ECG data from {input_csv}...")
ecg_data = []
try:
    with open(input_csv, "r") as file:
        reader = csv.reader(file)
        next(reader)  # Skip the header row
        for row in reader:
            try:
                ecg_data.append(float(row[2]))  # Assuming data is in the third column (index 2)
            except ValueError:
                print(f"Warning: Could not convert {row[2]} to float. Skipping this row.")
except FileNotFoundError:
    print(f"Error: The file {input_csv} was not found.")
    exit()
print(f"Total ECG data points read: {len(ecg_data)}")

# Step 3: Send ECG Data to ESP32 in chunks
fft_results = []
print(f"Sending ECG data to ESP32 in chunks of {SAMPLES}...")

plt.figure()
plt.xlabel("Frequency (Hz)")
plt.ylabel("Magnitude")
plt.title("FFT")

for i in range(0, len(ecg_data), SAMPLES):
    chunk = ecg_data[i:i + SAMPLES]
    if len(chunk) < SAMPLES:
        print(f"Warning: Remaining data chunk has less than {SAMPLES} samples, skipping.")
        break
    print(f"Sending chunk {i//SAMPLES + 1} of ECG data to ESP32...")

    # Send chunk to ESP32
    for value in chunk:
        try:
            ser.write(f"{value}\n".encode())  # Send ECG data point to ESP32
            time.sleep(0.008)  # Small delay to ensure data is sent
        except Exception as e:
            print(f"Error while sending data to ESP32: {e}")
            break

    # Step 4: Receive FFT Results from ESP32
    fft_chunk = []
    peaks_x = []
    peaks_y = []
    print("Receiving FFT results from ESP32...")

    # Receive FFT data (frequency, magnitude pairs)
    while True:
        line = ser.readline().decode().strip()
        if line == "Printing Peaks":
            break  # End of FFT data, peaks start here
        if line:
            try:
                frequency, magnitude = map(float, line.split(","))
                fft_chunk.append((frequency, magnitude))
            except ValueError:
                print(f"Warning: Could not parse FFT result line: {line}. Skipping this line.")

    # Receive Peaks Data
    while True:
        line = ser.readline().decode().strip()
        if line == "End":
            break  # End of peak data
        if line:
            try:
                frequency, magnitude = map(float, line.split(","))
                peaks_x.append(frequency)
                peaks_y.append(magnitude)
            except ValueError:
                print(f"Warning: Could not parse peak result line: {line}. Skipping this line.")

    # Extract FFT frequencies and magnitudes
    x = [freq for freq, _ in fft_chunk]
    y = [mag for _, mag in fft_chunk]

    # Detect valleys (local minima)
    peaks, _ = find_peaks(y, prominence=0.1, distance=5)  # Detect peaks
    valleys, _ = find_peaks(-np.array(y), prominence=0.1, distance=5)  # Detect valleys (negate signal)

    valleys_x = [x[v] for v in valleys]
    valleys_y = [y[v] for v in valleys]

    # Plot the FFT data
    plt.cla()
    plt.plot(x, y, color='b', label="FFT")  # FFT data in blue
    plt.scatter(peaks_x, peaks_y, color='r', label="Peaks")  # Peaks in red
    plt.scatter(valleys_x, valleys_y, color='g', label="Valleys")  # Valleys in green
    plt.xlabel("Frequency (Hz)")
    plt.ylabel("Magnitude")
    plt.xlim([0, 20])
    plt.title("FFT with Peaks and Valleys")
    plt.legend()
    plt.pause(0.1)  # Pause to update the plot

    fft_results.extend(fft_chunk)

plt.close()
